# EnerGIS Framework - Runner

Haupteinstiegspunkt für Optimierungsläufe mit dem EnerGIS Planning Framework.

## Übersicht

Dieses Notebook führt einen vollständigen Optimierungslauf durch:
- **Perfect Forecast (PF)**: Optimale Dimensionierung über den gesamten Zeitraum
- **Rolling Horizon (RH)**: Operative Planung mit rollendem Horizont
- **Model Predictive Control (MPC)**: RH mit Forecast-Updates
- **PF → RH/MPC**: Kombinierter Workflow mit Design-Fixierung

## Quick Start

1. Alle Zellen mit **Run All** ausführen
2. Bei Bedarf Config-Pfade in Zelle 3 anpassen
3. Ergebnisse werden automatisch in `saved_workflows/` gespeichert
4. Dashboard wird optional am Ende angezeigt

---

## 1. Setup & Imports

In [1]:
# Minimal-Bootstrap: Füge Projekt-Root zu sys.path hinzu
import sys
from pathlib import Path

# Finde Projekt-Root
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

# Auto-Setup mit notebook_helpers
from energis.io.notebook_helpers import setup_notebook_environment

PROJECT_ROOT = setup_notebook_environment()
print("\n✅ Setup abgeschlossen")

✅ Projekt-Root: c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat
✅ Matplotlib konfiguriert (Standard backend)
✅ Pandas konfiguriert
✅ Warnings unterdrückt

✅ Setup abgeschlossen


In [2]:
# Imports
from datetime import datetime
from energis.run import rolling_horizon as rh
from energis.io.notebook_helpers import (
    save_workflow_run,
    display_workflow_summary,
    display_kpi_summary
)

print("✅ Imports erfolgreich")

✅ Imports erfolgreich


## 2. Konfiguration

Die Konfiguration erfolgt über YAML-Dateien, die in der angegebenen Reihenfolge gemerged werden.
Spätere Dateien überschreiben frühere Einträge.

### Standard-Konfiguration:
- `base.yaml` - Basis-Einstellungen (Solver, Zeitschritt, etc.)
- `tech_catalog.yaml` - Technologie-Katalog (Komponenten-Definitionen)
- `default.site.yaml` - Standort-Daten (Input-Daten, Zeitzone, etc.)
- `baseline.system.yaml` - System-Topologie (Komponenten, Kapazitäten)
- `pf_then_rh.workflow.scenario.yaml` - Szenario (Run-Mode, RH-Parameter)

Passe die Config-Pfade nach Bedarf an!

In [3]:
# Konfigurationsdateien
CONFIG_PATHS = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/stadtbach_brownfield.system.yaml',
    'configs/scenarios/stadtbach_brownfield.scenario.yaml',
]


# Optional: Overrides für spezifische Parameter
# Beispiele:
# - Run-Mode ändern: {'scenario': {'run_mode': 'PF_ONLY'}}
# - Solver ändern: {'run': {'solver': 'glpk'}}
# - RH-Parameter: {'scenario': {'rolling_horizon': {'heat_horizon_hours': 72}}}
OVERRIDES = None

# Config-Dateien prüfen
print("📋 Konfigurationsdateien:")
all_exist = True
for cfg_path in CONFIG_PATHS:
    full_path = PROJECT_ROOT / cfg_path
    exists = full_path.exists()
    symbol = '✅' if exists else '❌'
    print(f"  {symbol} {cfg_path}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("Nicht alle Config-Dateien gefunden!")

print("\n✅ Konfiguration OK")

📋 Konfigurationsdateien:
  ✅ configs/base.yaml
  ✅ configs/tech_catalog.yaml
  ✅ configs/sites/default.site.yaml
  ✅ configs/systems/stadtbach_brownfield.system.yaml
  ✅ configs/scenarios/stadtbach_brownfield.scenario.yaml

✅ Konfiguration OK


## 3. Workflow ausführen

Der Workflow führt die Optimierung gemäß der konfigurierten Run-Mode aus:

- **PF_ONLY**: Nur Perfect Forecast
- **RH_ONLY**: Nur Rolling Horizon
- **MPC_ONLY**: Nur Model Predictive Control (mit Forecasts)
- **PF_THEN_RH**: PF für Dimensionierung, dann RH mit fixiertem Design
- **PF_THEN_MPC**: PF für Dimensionierung, dann MPC mit fixiertem Design

In [ ]:
%%time
print("="*70)
print("🚀 STARTE OPTIMIERUNG")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS, overrides=OVERRIDES)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_success = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER")
    print("="*70)
    print(f"\nFehler: {e}\n")
    
    import traceback
    traceback.print_exc()
    
    workflow = None
    optimization_success = False

🚀 STARTE OPTIMIERUNG
Start: 2025-12-16 13:59:17

[LOAD] Import_Data.xlsx → 8760 Schritte von 2023-01-01 00:00:00 bis 2023-12-31 23:00:00
[SCENARIO] Zeitraum 2023-01-01 00:00:00 → 2023-03-01 00:00:00 (1417 Schritte)
[BUILD] Using simple storage (single-zone model)
[BUILD] #el_in=5, #el_out=3, #ht_out=12, #ht_in=1


## 4. Workflow speichern & exportieren

Speichert den Workflow mit allen Ergebnissen, Metadaten und Plots in `saved_workflows/`.

In [ ]:
if optimization_success and workflow:
    # Workflow-Name und Beschreibung (anpassbar)
    WORKFLOW_NAME = "Baseline Simulation"
    WORKFLOW_DESCRIPTION = "PF + RH Optimierung mit Standard-Konfiguration"
    
    # Workflow speichern (inkl. CSV, PDF, SVG Exports)
    workflow_dir = save_workflow_run(
        workflow,
        name=WORKFLOW_NAME,
        description=WORKFLOW_DESCRIPTION,
        config_paths=CONFIG_PATHS
    )
    
    print(f"\n💡 Dashboard anzeigen:")
    print(f"   • In diesem Notebook: Siehe Zelle 7")
    print(f"   • In interactive_dashboard.ipynb: Workflow auswählen")
    
else:
    print("⚠️  Workflow-Speicherung übersprungen (Optimierung fehlgeschlagen)")

📦 Exportiere Ergebnisse (CSV, PDF, SVG)...
📁 Speicherverzeichnis: exports\20251216_114321_stadtbach-brownfield
💾 Speichere Workflow-Objekt...
✅ Workflow gespeichert: workflow.pkl
📝 Erstelle Metadaten...
✅ Metadaten gespeichert: metadata.json
🔄 Verschiebe nach saved_workflows/...
✅ Verschoben nach: c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat\notebooks\saved_workflows\20251216_114321_stadtbach-brownfield

✅ WORKFLOW ERFOLGREICH GESPEICHERT
📂 Verzeichnis: c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat\notebooks\saved_workflows\20251216_114321_stadtbach-brownfield
📊 Dateien:
   • workflow.pkl    - Workflow-Objekt
   • metadata.json   - Metadaten
   • *.csv           - Zeitreihen
   • *.pdf, *.svg    - Plots
   • design.json     - Anlagen-Design

💡 Dashboard anzeigen:
   • In diesem Notebook: Siehe Zelle 7
   • In interactive_dashboard.ipynb: Workflow auswählen


## 5. Zusammenfassung

Zeigt die wichtigsten Kennzahlen aus dem Optimierungslauf.

In [ ]:
results = solver.solve(model)
print(model.results())
print(model.display())
print(results)

if optimization_success and workflow:
    # Zusammenfassung anzeigen
    display_workflow_summary(workflow)
    
else:
    print("⚠️  Keine Ergebnisse verfügbar")


📊 WORKFLOW-ZUSAMMENFASSUNG

🔄 Workflow: PF

📦 Verfügbare Ergebnisse:
   ✅ Perfect Forecast (PF)
   ❌ Rolling Horizon (RH)
   ❌ Model Predictive Control (MPC)

💰 Kosten:
   Gesamtkosten:       13,703,937 EUR
   • Brennstoff:       12,731,448 EUR  ( 92.9%)
   • CAPEX:             1,332,569 EUR  (  9.7%)

🏭 Anlagen-Design:
   Wärmepumpen:
      HP1:   0.00 MW
      HP2:   0.00 MW
      HP3:  50.24 MW
      HP4:   0.00 MW
   Speicher:     0.00 MWh



NameError: name 'results' is not defined

## 6. Key Performance Indicators

Detaillierte KPI-Analyse mit Kostenaufschlüsselung und Komponentenauslastung.

In [ ]:
if optimization_success and workflow:
    # Detaillierte KPI-Analyse
    display_kpi_summary(workflow)
else:
    print("⚠️  Keine KPIs verfügbar")


📊 KEY PERFORMANCE INDICATORS

💰 Wirtschaftlichkeit:
   Gesamtkosten:       13,703,937 EUR

   💶 Detaillierte Aufschlüsselung:
      Brennstoffkosten:           12,731,448 EUR
      Investitionskosten:          1,332,569 EUR

⚡ Elektrische Energie:
   Netzbezug:                   0 MWh
   Einspeisung:            26,712 MWh

🏭 Komponenten-Auslastung:

   Wärmepumpen:
      HP1     : Ø   0.00 MW | Max   0.00 MW |     0 h aktiv
      HP2     : Ø   0.00 MW | Max   0.00 MW |     0 h aktiv
      HP3     : Ø  15.14 MW | Max  50.24 MW |  3963 h aktiv
      HP4     : Ø   0.00 MW | Max   0.00 MW |     0 h aktiv

   Speicher:
      Max SOC:             0.00 MWh
      Ø SOC:               0.00 MWh



In [ ]:
if optimization_success and workflow and thermal_network_enabled:
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        import pandas as pd
        
        # Get network time series
        result = workflow.pf_result or workflow.rh_result or workflow.mpc_result
        
        if result and hasattr(result, 'series'):
            # Filter network series (all keys starting with NET_)
            network_series = {k: v for k, v in result.series.items() if k.startswith('NET_')}
            
            if network_series:
                # Get timestamps
                timestamps = list(result.table.index)
                
                # Create subplots
                fig = make_subplots(
                    rows=2, cols=1,
                    subplot_titles=('Temperaturen an Knoten', 'Wärmeverluste in Rohren'),
                    vertical_spacing=0.12
                )
                
                # Plot 1: Node temperatures
                temp_keys = [k for k in network_series.keys() if '_T_supply_C' in k or '_T_return_C' in k]
                for key in temp_keys[:6]:  # Limit to 6 lines for readability
                    node_id = key.replace('NET_', '').replace('_T_supply_C', '').replace('_T_return_C', '')
                    line_type = 'supply' if '_T_supply_C' in key else 'return'
                    fig.add_trace(
                        go.Scatter(x=timestamps, y=network_series[key], 
                                  name=f"{node_id} ({line_type})",
                                  mode='lines'),
                        row=1, col=1
                    )
                
                # Plot 2: Heat losses
                loss_keys = [k for k in network_series.keys() if '_Q_loss_supply_kW' in k]
                for key in loss_keys[:6]:  # Limit to 6 lines
                    pipe_id = key.replace('NET_', '').replace('_Q_loss_supply_kW', '')
                    fig.add_trace(
                        go.Scatter(x=timestamps, y=network_series[key],
                                  name=pipe_id,
                                  mode='lines'),
                        row=2, col=1
                    )
                
                # Update layout
                fig.update_xaxes(title_text="Zeit", row=2, col=1)
                fig.update_yaxes(title_text="Temperatur [°C]", row=1, col=1)
                fig.update_yaxes(title_text="Verluste [kW]", row=2, col=1)
                fig.update_layout(height=700, showlegend=True, title_text="Thermisches Netzwerk - Zeitverlauf")
                
                fig.show()
                
                print(f"\n✅ {len(temp_keys)} Temperaturverläufe und {len(loss_keys)} Verlustprofile visualisiert")
            else:
                print("ℹ️  Keine Netzwerk-Zeitreihen verfügbar für Visualisierung")
    except ImportError:
        print("⚠️  plotly nicht installiert. Installiere mit: pip install plotly")
    except Exception as e:
        print(f"⚠️  Visualisierung fehlgeschlagen: {e}")

NameError: name 'thermal_network_enabled' is not defined

### 7.1 Netzwerk-Visualisierung

Zeigt Temperaturverläufe und Verluste im Netzwerk über die Zeit.

In [ ]:
if optimization_success and workflow:
    # Check if thermal network was enabled
    thermal_network_enabled = workflow.config.get('thermal_network', {}).get('enabled', False)
    
    if thermal_network_enabled:
        print("🌡️  THERMISCHES NETZWERK AKTIVIERT")
        print("=" * 70)
        
        # Get the appropriate result object
        result = None
        if workflow.pf_result:
            result = workflow.pf_result
            result_type = "PF"
        elif workflow.rh_result:
            result = workflow.rh_result
            result_type = "RH"
        elif workflow.mpc_result:
            result = workflow.mpc_result
            result_type = "MPC"
        
        if result and hasattr(result, 'summary') and 'thermal_network' in result.summary:
            net_summary = result.summary['thermal_network']
            
            print(f"\n📊 Netzwerk-Zusammenfassung ({result_type}):")
            print(f"  Knoten:              {net_summary.get('Number_of_nodes', 0)}")
            print(f"  Rohrleitungen:       {net_summary.get('Number_of_pipes', 0)}")
            print(f"  Gesamtlänge:         {net_summary.get('Total_pipe_length_m', 0):.0f} m")
            print(f"\n🔥 Wärmelieferung:")
            print(f"  Geliefert:           {net_summary.get('Total_heat_delivered_MWh', 0):.1f} MWh")
            print(f"  Verluste:            {net_summary.get('Total_heat_loss_MWh', 0):.1f} MWh")
            print(f"  Verlustrate:         {net_summary.get('Heat_loss_percentage', 0):.2f}%")
            
            # Check if losses are in reasonable range
            loss_pct = net_summary.get('Heat_loss_percentage', 0)
            if 0.5 <= loss_pct <= 1.5:
                print(f"  ✅ Verluste im typischen Bereich für moderne Fernwärmenetze (0.5-1.5%)")
            elif loss_pct > 1.5:
                print(f"  ⚠️  Verluste höher als typisch - Isolierung prüfen?")
            else:
                print(f"  ✅ Sehr gute Netzeffizienz!")
                
        else:
            print("\n⚠️  Netzwerk-Ergebnisse nicht verfügbar")
            print("    (Möglicherweise wurde das Modell nicht vollständig gelöst)")
    else:
        print("ℹ️  Thermisches Netzwerk ist nicht aktiviert")
        print("   Um das Netzwerk zu nutzen, füge in der Szenario-Konfiguration hinzu:")
        print("   thermal_network:")
        print("     enabled: true")
        print("     topology_file: stadtbach_network.yaml")
else:
    print("⚠️  Keine Netzwerk-Ergebnisse verfügbar")

ℹ️  Thermisches Netzwerk ist nicht aktiviert
   Um das Netzwerk zu nutzen, füge in der Szenario-Konfiguration hinzu:
   thermal_network:
     enabled: true
     topology_file: stadtbach_network.yaml


## 7. Thermal Network Results 🌡️

Wenn das thermische Netzwerk aktiviert ist, werden hier die Netzwerk-spezifischen Ergebnisse angezeigt:
- Wärmeverluste im Netz
- Temperaturen an Knoten
- Durchflussraten in Rohren

## 7. Dashboard zur Visualisierung

Um die gespeicherten Simulationsergebnisse zu visualisieren, verwende das **Standalone Dashboard**.

### 🎛️ Dashboard starten:

**Option 1: Python-Skript (empfohlen)**
```bash
python start_dashboard.py
```

**Option 2: Workflow Browser Notebook**
```bash
panel serve notebooks/workflow_browser.ipynb --show
```

Das Dashboard läuft unabhängig von der Simulation und lädt automatisch alle gespeicherten Workflows aus `saved_workflows/`.

In [ ]:
print("📊 Simulationsergebnisse wurden gespeichert!")
print("\n🎛️ Zum Visualisieren starte das Dashboard:")
print("   python start_dashboard.py")
print("\nOder verwende das Workflow Browser Notebook:")
print("   panel serve notebooks/workflow_browser.ipynb --show")
print("\n💡 Das Dashboard lädt automatisch alle Workflows aus 'saved_workflows/'")

📊 Simulationsergebnisse wurden gespeichert!

🎛️ Zum Visualisieren starte das Dashboard:
   python start_dashboard.py

Oder verwende das Workflow Browser Notebook:
   panel serve notebooks/workflow_browser.ipynb --show

💡 Das Dashboard lädt automatisch alle Workflows aus 'saved_workflows/'


---

## 📚 Weitere Informationen

- **Dokumentation**: `README.md`, `ARCHITECTURE_V2.md`
- **Methodologie**: `docs/methodology.md`
- **CLI-Nutzung**: `python -m energis.run.rolling_horizon --help`
- **Andere Notebooks**:
  - `interactive_dashboard.ipynb` - Dashboard mit gespeicherten Workflows
  - `scenario_studio.ipynb` - Interaktive Szenario-Analyse

## 🌐 Dashboard als Webapp

Um das Dashboard als eigenständige Webapp zu starten:

```bash
panel serve runner.ipynb --show
# Oder auf spezifischem Port:
panel serve runner.ipynb --port 5006 --show
```

---